In [1]:
!pip install wordfreq
import pandas as pd
import re
import gc
from tqdm import tqdm
import ujson as json
import csv
from nltk.corpus import stopwords
from wordfreq import top_n_list

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00


In [2]:
# Load only the necessary columns for efficiency
df = pd.read_csv("/kaggle/input/german-parliament/speeches_all.csv", usecols=['id', 'begin', 'end', 'sentence'])
gc.collect()

# Rename column
df.rename(columns={'sentence': 'text'}, inplace=True)

# Initialize tqdm for progress bars
tqdm.pandas()


def preprocess(df, length_threshold):
    stopwords_set = set(stopwords.words('german'))
    custom_stopwords = {"ja", "nur", "uns", "ist", "tun", "sei", "wollen", "aber", "machen", "könnte", "dm", "dr", "dem", "vor", "aus"}
    stopwords_set.update(custom_stopwords)

    def remove_special_characters(text):
        pattern = r'[^a-zA-Z0-9äöüÄÖÜß\s]'
        clean_text = re.sub(pattern, '', text)
        return clean_text

    def remove_stopwords(text, stopwords_set):
        mostly_numeric_pattern = r"^\d*[a-zA-Z]?\d*$"
        return ' '.join([word for word in text.split() if word.lower() not in stopwords_set and len(word) > 2 and not re.match(mostly_numeric_pattern, word)])

    df['text'] = df['text'].astype(str)
    df['text'].replace(to_replace=r"\.\.+", value=" ", regex=True, inplace=True)
    df['text'].replace(to_replace=r"\-\-+", value=" ", regex=True, inplace=True)
    df['text'].replace(to_replace=r"__+", value=" ", regex=True, inplace=True)
    df['text'].replace(to_replace=r"\*\*+", value=" ", regex=True, inplace=True)
    df['text'].replace(to_replace=r"\s+", value=" ", regex=True, inplace=True)
    df['text'] = df['text'].progress_apply(remove_special_characters)
    df['text'] = df['text'].progress_apply(lambda x: remove_stopwords(x, stopwords_set))
    df['length'] = df['text'].progress_apply(lambda x: len(x.split()))
    df = df[df['length'] > length_threshold]
    print(f"Average Speech length: {[df['length'].mean()]}")

    return df
    
# Apply preprocessing
df = preprocess(df, 10)
gc.collect()

# Optimize sentence splitting
speech_list = df["text"].tolist()
sentences = [speech.split() for speech in tqdm(speech_list)]

# Efficient JSON dumping using ujson
with open('training_corpus.json', 'w') as f:
    json.dump(sentences, f)

/tmp/ipykernel_17/2110289920.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['text'].replace(to_replace=r"\.\.+", value=" ", regex=True, inplace=True)
/tmp/ipykernel_17/2110289920.py:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(v

Average Speech length: [39.39750727145982]


100%|██████████| 2912070/2912070 [00:38<00:00, 75680.23it/s]
